# DORA Incident Agent (Rheinbank, synthetic data)

Checks whether an IT incident at the made-up bank Rheinbank is "major" under DORA. The AI collects facts from simulated bank systems; fixed rules from the law decide; a person approves the draft notification. Nothing is ever sent. **Work in progress.**

## How to run
1. **Add your Groq key once.** Click the key icon in the left sidebar (Secrets). Add a secret named exactly `GROQ_API_KEY` with your own Groq key as the value, and switch **Notebook access** on. Never paste the key into a cell.
2. **Run the cells in order**, top to bottom: setup, key, then start the page.
3. The live tests at the bottom are optional. They use real Groq calls and your token quota.

## Troubleshooting
- **The page stays blank, or says it refused to connect.** Chrome is blocking the cookies the page needs. Open `chrome://settings/cookies` (paste it into the address bar), find **Allowed to use third-party cookies**, click **Add**, and add both `[*.]colab.dev` and `[*.]googleusercontent.com`. Then reload this tab and run the launch cell again.
- **You want the newest version of the code.** Use **Runtime → Disconnect and delete runtime**, then run all cells again from the top. Re-running the setup cell in the same runtime does not fetch new code.
- **The page froze, or you closed it.** Re-run **only the launch cell**. It closes the old server first. Don't re-run the setup cell in the same runtime.
- **"No secret named GROQ_API_KEY" or "no notebook access".** Check step 1: the name must be exactly `GROQ_API_KEY`, and the **Notebook access** switch next to it must be on. Then run the key cell again.
- **Access refused, 403, or connection errors from Groq.** A VPN or a company network can block Groq. Switch the VPN off and run the cell again. Also check that your key is still valid in your Groq account.

In [ ]:
# Setup: get the code and install the pinned requirements (exact versions, see requirements.txt).
# Runs once per runtime. For a newer version of the code: Runtime -> Disconnect and delete runtime.
import importlib.metadata
import os
import subprocess
import sys
import time

REPO_URL = "https://github.com/samuel502/DORA-Incident-Agent.git"
REPO_DIR = "/content/DORA-Incident-Agent"
# Packages Colab may already have loaded in an older version. If one changes, Python keeps the old one.
CHECK = ["numpy", "pandas", "pydantic", "httpx", "gradio", "fastapi", "starlette", "PIL"]
DISTRIBUTION = {"PIL": "pillow"}

started = time.monotonic()
if os.path.isdir(REPO_DIR):
    print("Code already here from earlier in this runtime. Not fetching again.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

loaded = {name: getattr(sys.modules[name], "__version__", None) for name in CHECK if name in sys.modules}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "-r",
                f"{REPO_DIR}/requirements.txt"], check=True)
changed = sorted(name for name, version in loaded.items()
                 if version != importlib.metadata.version(DISTRIBUTION.get(name, name)))

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.chdir(REPO_DIR)
print(f"Setup done in {time.monotonic() - started:.0f} seconds.")
if changed:
    print("RESTART NEEDED: Colab had already loaded an older " + ", ".join(changed) + ". Use Runtime -> "
          "Restart session, then run all cells again from the top. The second setup is quick.")

In [ ]:
# Key: read GROQ_API_KEY from YOUR Colab Secrets into this runtime's memory only.
# It is never printed, never written to a file, and gone when the runtime ends.
import os
from google.colab import userdata

_key = None
try:
    _key = userdata.get("GROQ_API_KEY")
except userdata.SecretNotFoundError:
    print("No secret named GROQ_API_KEY. Add it in the Secrets panel (key icon, left sidebar).")
except userdata.NotebookAccessError:
    print("The secret GROQ_API_KEY exists, but this notebook has no access. Switch on 'Notebook access' next to it.")
except Exception as error:  # only the error type is shown, never its text
    print(f"Could not read the secret ({type(error).__name__}). See Troubleshooting at the top.")

if _key and _key.strip():
    os.environ["GROQ_API_KEY"] = _key.strip()
    print("Key loaded into this runtime's memory. It is not shown and not saved.")
elif _key is not None:
    print("The secret GROQ_API_KEY is empty. Paste your key as its value in the Secrets panel.")
del _key

## Start the page
Run the cell below. The page appears under it. If it stays blank, see Troubleshooting at the top (cookies). If the page freezes, re-run only this cell.

In [ ]:
# Start the page and show it below. No public link: only you can open it, through Colab's own forwarding.
# Re-run only this cell to restart the page: it closes the old server first.
from dora_agent.ui.launch import launch

launch()

## Optional: live tests with the real Groq model
Runs sample cases through the agent with real Groq calls. It uses your token quota, and each case pauses 60 seconds to stay under Groq's free-tier limits, so allow a few minutes per case. Leave `LIVE_CASES` empty to run all 27 (over an hour). The report goes to `logs/live/` in this runtime and is gone when the runtime ends.

In [ ]:
# Live tests: opt-in, real Groq calls, key from the key cell above (never printed).
import os
import subprocess
import sys

LIVE_CASES = "C01,N01,B03"   # comma-separated case IDs; empty = all 27
LIVE_PAUSE_SECONDS = "60"    # pause between cases (Groq free tier)

if not os.environ.get("GROQ_API_KEY", "").strip():
    print("No key loaded. Run the key cell first.")
else:
    env = dict(os.environ, RUN_LIVE_TESTS="1", LIVE_CASES=LIVE_CASES, LIVE_PAUSE_SECONDS=LIVE_PAUSE_SECONDS)
    command = [sys.executable, "-m", "pytest", "tests/live", "-s", "-q"]
    with subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:  # show progress as it happens
            print(line, end="")
    print(f"\nLive tests finished with exit code {process.returncode} (0 = all passed).")